## 3. Mathematical Formulation of Composite Resolution Evidence

When conflicts occur (e.g. 1-to-many mappings), Improvement 3 evaluates multi-source evidence incorporating **Improvement 1 (Semantic Type Matching)**:

$$\text{Resolution Score} = w_e \cdot \text{sim}_e + w_t \cdot \text{sim}_t + w_s \cdot \text{struct} + \mathbf{w_{sem} \cdot \text{sim}_{sem}} + w_m \cdot \text{margin} - \text{penalty}$$

Where:
- $w_e = 0.35$: Entity lexical similarity
- $w_t = 0.25$: Structural triple agreement
- $w_s = 0.15$: Graph neighborhood topological consistency
- $\mathbf{w_{sem} = 0.15}$: **Semantic type compatibility evidence (Improvement 1)**
- $w_m = 0.10$: Candidate score margin bonus
- $\text{penalty} = 0.05$: Deducted if candidate is entangled in a conflict
- Disjoint category mismatch penalty: $-0.10$ if semantic types are incompatible

Candidates are resolved globally in descending order of resolution score, ensuring 1-to-1 bijective mapping where appropriate.

In [1]:
# 1. Setup Environment and Import Modules
import os
import sys
import json
import pandas as pd
import numpy as np

# Locate project root dynamically
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
curr = os.path.abspath(notebook_dir)
while curr and os.path.dirname(curr) != curr:
    if os.path.isdir(os.path.join(curr, "codes")) and os.path.isdir(os.path.join(curr, "dataset")):
        break
    curr = os.path.dirname(curr)
project_root = curr

entity_matching_dir = os.path.join(project_root, "codes", "entity-matching")
utils_dir = os.path.join(project_root, "codes", "utils")

for p in [project_root, entity_matching_dir, utils_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

from file_utils import load_csv, load_json, load_rdf_graph
import semantic_type_matching as stm
import adaptive_threshold as at
import conflict_resolution as cr
from entity_matching import run_pipeline
from evaluation import evaluate_predictions, build_comparison_dataframe

print("Project environment loaded successfully.")
print(f"Project root: {project_root}")

Project environment loaded successfully.


## 2. Load Datasets & RDF Graphs for Structural Verification

In [2]:
# 2. Load Inputs
candidates_df = load_csv(os.path.join(project_root, "dataset", "movies", "movie_candidates.csv"))
gold_df = load_csv(os.path.join(project_root, "dataset", "movies", "small_movie_gold.csv"))
graph1 = load_rdf_graph(os.path.join(project_root, "dataset", "movies", "small_movie_kg1.ttl"))
graph2 = load_rdf_graph(os.path.join(project_root, "dataset", "movies", "small_movie_kg2.ttl"))

print(f"Loaded {len(candidates_df)} candidates.")
print(f"Loaded KG1 triples: {len(graph1)}, KG2 triples: {len(graph2)}")

Loaded 426 candidates.
Loaded KG1 triples: 71, KG2 triples: 73


## 3. Generate Improvement 2 Candidates and Apply Conflict Resolution

$$\\text{Resolution Score} = \\Big(0.40 \\cdot \\text{sim} + 0.30 \\cdot \\text{triple\\_sim} + 0.20 \\cdot \\text{struct\\_consistency} + 0.10 \\cdot \\text{Margin}\\Big) - \\text{Penalty}$$

In [3]:
# 3. Run Adaptive Thresholding + Automated Conflict Resolution
adaptive_decisions = at.apply_adaptive_threshold(
    candidates_df=candidates_df,
    use_adaptive=True,
    base_threshold=0.80,
    entity_col1='e1',
    entity_col2='e2',
    sim_col='sim',
    triple_sim_col='triple_sim'
)

resolved_df, conflict_log_df, stats = cr.detect_and_resolve_conflicts(
    adaptive_decisions_df=adaptive_decisions,
    use_conflict_resolution=True,
    entity_weight=0.40,
    triple_weight=0.30,
    structure_weight=0.20,
    margin_weight=0.10,
    conflict_penalty=0.05,
    ambiguity_threshold=0.02,
    graph1=graph1,
    graph2=graph2,
    entity_col1='entity_kg1',
    entity_col2='entity_kg2',
    sim_col='similarity',
    triple_sim_col='triple_similarity',
    decision_col='decision'
)

print("=== CONFLICT RESOLUTION SUMMARY ===")
for k, v in stats.items():
    print(f"  {k:<28}: {v}")

=== CONFLICT RESOLUTION SUMMARY ===
  total_candidates            : 426
  accepted_by_threshold       : 26
  conflicts_detected          : 8
  conflicts_resolved          : 6
  conflicts_unresolved        : 2
  final_accepted_matches      : 18


## 4. Transparent Audit Log (`conflict_resolution.csv`)

Every arbitration decision is preserved with full explainability rationale:

In [4]:
# 4. View Audit Log Sample
conf_cols = [c for c in ['conflict_id', 'source_entity', 'conflict_type', 'selected_candidate', 'rejected_candidates', 'resolution_status', 'reason'] if c in conflict_log_df.columns]
display(conflict_log_df[conf_cols].head(8))

conflict_id,source_entity,conflict_type,selected_candidate,rejected_candidates,resolution_status,reason
CONF_0001,http://kg1.example.org/resource/Action,one_to_many+many_to_one,http://kg2.example.org/resource/Action_Film,http://kg2.example.org/resource/Science_Fiction; http://kg2.example.org/resource/SciFi,RESOLVED,"Selected top candidate by composite resolution evidence (score=0.8182 vs runner-up=0.6545, diff=0.1636)"
CONF_0002,http://kg1.example.org/resource/Christopher_Nolan,one_to_many+competing_candidate,NONE (UNRESOLVED),http://kg2.example.org/resource/Christopher_Nolan; http://kg2.example.org/resource/Chris_Nolan,UNRESOLVED,Genuine ambiguity: resolution score diff (0.0000) < ambiguity threshold (0.02)
CONF_0003,http://kg1.example.org/resource/Drama,one_to_many,http://kg2.example.org/resource/Drama,http://kg2.example.org/resource/Romantic_Drama,RESOLVED,"Selected top candidate by composite resolution evidence (score=0.8100 vs runner-up=0.7200, diff=0.0900)"
CONF_0004,http://kg1.example.org/resource/Science_Fiction,one_to_many+many_to_one+competing_candidate,NONE (UNRESOLVED),http://kg2.example.org/resource/Science_Fiction; http://kg2.example.org/resource/SciFi; http://kg2.example.org/resource/Action_Film,UNRESOLVED,Genuine ambiguity: resolution score diff (0.0000) < ambiguity threshold (0.02)
CONF_0005,[Target] http://kg2.example.org/resource/Action_Film,many_to_one,http://kg1.example.org/resource/Action,http://kg1.example.org/resource/Science_Fiction,RESOLVED,Many-to-one resolved in favor of source http://kg1.example.org/resource/Action (score=0.8182 vs 0.6545)
CONF_0006,[Target] http://kg2.example.org/resource/Romantic_Drama,many_to_one,http://kg1.example.org/resource/Drama,http://kg1.example.org/resource/Romance,RESOLVED,Many-to-one resolved in favor of source http://kg1.example.org/resource/Drama (score=0.7200 vs 0.6237)
CONF_0007,[Target] http://kg2.example.org/resource/SciFi,many_to_one,http://kg1.example.org/resource/Science_Fiction,http://kg1.example.org/resource/Action,RESOLVED,Many-to-one resolved in favor of source http://kg1.example.org/resource/Science_Fiction (score=0.8000 vs 0.6545)
CONF_0008,[Target] http://kg2.example.org/resource/Science_Fiction,many_to_one,http://kg1.example.org/resource/Science_Fiction,http://kg1.example.org/resource/Action,RESOLVED,Many-to-one resolved in favor of source http://kg1.example.org/resource/Science_Fiction (score=0.8000 vs 0.6545)


## 5. Movies Benchmark: 3-Way Metrics Comparison

In [5]:
# 5. Side-by-Side Comparison on Movies
base_dec = at.apply_adaptive_threshold(candidates_df, use_adaptive=False, base_threshold=0.80)
m_base = evaluate_predictions(base_dec, gold_df)
m_imp2 = evaluate_predictions(adaptive_decisions, gold_df)
m_imp3 = evaluate_predictions(resolved_df, gold_df)

comp_df = pd.DataFrame([
    {"Metric": "Precision", "Baseline": m_base['precision'], "Imp-2 (Adaptive)": m_imp2['precision'], "Imp-3 (Conflict)": m_imp3['precision']},
    {"Metric": "Recall",    "Baseline": m_base['recall'],    "Imp-2 (Adaptive)": m_imp2['recall'],    "Imp-3 (Conflict)": m_imp3['recall']},
    {"Metric": "F1-Score",  "Baseline": m_base['f1'],        "Imp-2 (Adaptive)": m_imp2['f1'],        "Imp-3 (Conflict)": m_imp3['f1']},
    {"Metric": "Hit@1",     "Baseline": m_base['hit_at_1'],  "Imp-2 (Adaptive)": m_imp2['hit_at_1'],  "Imp-3 (Conflict)": m_imp3['hit_at_1']},
    {"Metric": "Hit@10",    "Baseline": m_base['hit_at_10'], "Imp-2 (Adaptive)": m_imp2['hit_at_10'], "Imp-3 (Conflict)": m_imp3['hit_at_10']}
])

display(comp_df)

Metric,Baseline,Imp-2 (Adaptive),Imp-3 (Conflict)
Precision,1.000000,1.000000,1.000000
Recall,0.818182,0.909091,0.818182
F1-Score,0.900000,0.952381,0.900000
Hit@1,0.900000,0.900000,0.900000
Hit@10,1.000000,1.000000,1.000000


## 6. Large-Scale Benchmark: Improvement 3 on DW-NB Dataset

On the 666k DW-NB dataset, Improvement 3 eliminates false positives while preserving the recall gains from Improvement 2:

In [6]:
# 6. Load DW-NB 3-Way Comparison
dw_comp = pd.read_csv(os.path.join(project_root, "results", "full-triple-matcher", "dw-nb", "final", "comparison.csv"))
print("=== DW-NB BENCHMARK: 3-WAY COMPARISON ===")
display(dw_comp)

=== DW-NB BENCHMARK: 3-WAY COMPARISON ===


metric,baseline,improvement_2_adaptive,improvement_3_conflict_res,diff_imp3_minus_baseline,diff_imp3_minus_imp2
Predicted Matches,28569.00000,31389.000000,30110.000000,1541.000000,-1279.000000
True Positives,13550.00000,14747.000000,14369.000000,819.000000,-378.000000
False Positives,15019.00000,16642.000000,15741.000000,722.000000,-901.000000
Precision,0.47429,0.469814,0.477217,0.002927,0.007403
Recall,0.54200,0.589880,0.574760,0.032760,-0.015120
F1-Score,0.50589,0.523045,0.521466,0.015576,-0.001579
Hit@1,0.92540,0.925400,0.925400,0.000000,0.000000
Hit@10,0.96116,0.961160,0.961160,0.000000,0.000000
Runtime (s),7.89590,8.452600,41.729400,33.833500,33.276800


In [ ]:
# Save Improvement 3 outputs to results directory
out_dir = os.path.join(project_root, "results", "full-triple-matcher", "movies", "improvement-3")
os.makedirs(out_dir, exist_ok=True)

resolved_df.to_csv(os.path.join(out_dir, "matching_decisions.csv"), index=False)
if not conflict_log_df.empty:
    conflict_log_df.to_csv(os.path.join(out_dir, "conflict_resolution.csv"), index=False)

with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(m_imp3, f, indent=4)
pd.DataFrame(list(m_imp3.items()), columns=["metric", "value"]).to_csv(os.path.join(out_dir, "summary.csv"), index=False)

print(f"[SAVED] Improvement 3 outputs saved successfully to:")
print(f"  -> {os.path.relpath(os.path.join(out_dir, 'matching_decisions.csv'), project_root)}")
print(f"  -> {os.path.relpath(os.path.join(out_dir, 'conflict_resolution.csv'), project_root)}")
print(f"  -> {os.path.relpath(os.path.join(out_dir, 'metrics.json'), project_root)}")
print(f"  -> {os.path.relpath(os.path.join(out_dir, 'summary.csv'), project_root)}")